In [ ]:
# ==============================================================================
# 1. ENVIRONMENT SETUP & WORKSPACE INITIALIZATION
# ==============================================================================

!pip install -q \
    chromadb==1.5.9 \
    sentence-transformers \
    mitreattack-python \
    transformers \
    accelerate \
    peft \
    bitsandbytes \
    numpy==1.26.4 \
    sentencepiece

import os
import sys
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Configure workspace directories and stage dataset assets
sys.path.append("/kaggle/input/datasets/karthiksethuraman6/qwen-atlas-data")
!cp /kaggle/input/datasets/karthiksethuraman6/qwen-atlas-data/enterprise-attack.json .
!cp /kaggle/input/datasets/karthiksethuraman6/qwen-atlas-data/index_mappings.json .

import chroma_rag
print("Environment successfully initialized.")

In [ ]:
# Populate ChromaDB collections with structural MITRE intelligence
print("Ingesting threat data collections...")
chroma_rag.ingest_techniques()
chroma_rag.ingest_groups()
chroma_rag.ingest_mitigations()

# Load quantized causal model parameters
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
LORA_MODEL = "Karthik-sr/qwen-atlas-raft-v1"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, LORA_MODEL)
model.eval()

print("\nModel and Knowledge base initialization complete.")

In [ ]:
QUERIES = [
    # A. Technique ID Lookup
    {"id": "A1", "query": "What is T1003?",        "category": "Technique ID Lookup"},
    {"id": "A2", "query": "What is T1055?",        "category": "Technique ID Lookup"},
    {"id": "A3", "query": "What is T1195.002?",    "category": "Technique ID Lookup"},
    {"id": "A4", "query": "What is T1110.003?",    "category": "Technique ID Lookup"},
    {"id": "A5", "query": "What is T1021.001?",    "category": "Technique ID Lookup"},
    # B. Technique Name Lookup
    {"id": "B1", "query": "What is Process Injection?",      "category": "Technique Name Lookup"},
    {"id": "B2", "query": "What is OS Credential Dumping?",  "category": "Technique Name Lookup"},
    {"id": "B3", "query": "Explain Password Spraying.",      "category": "Technique Name Lookup"},
    {"id": "B4", "query": "What is PowerShell?",             "category": "Technique Name Lookup"},
    {"id": "B5", "query": "What is Windows Command Shell?",  "category": "Technique Name Lookup"},
    # C. Group Information
    {"id": "C1", "query": "What is APT29?",        "category": "Group Information"},
    {"id": "C2", "query": "What is APT33?",        "category": "Group Information"},
    {"id": "C3", "query": "What is Lazarus Group?","category": "Group Information"},
    {"id": "C4", "query": "What is FIN7?",         "category": "Group Information"},
    {"id": "C5", "query": "What is Kimsuky?",      "category": "Group Information"},
    # D. Group to Techniques
    {"id": "D1", "query": "What techniques does APT29 use?",        "category": "Group to Techniques"},
    {"id": "D2", "query": "What techniques does Cozy Bear use?",    "category": "Group to Techniques"},
    {"id": "D3", "query": "What techniques does The Dukes use?",    "category": "Group to Techniques"},
    {"id": "D4", "query": "What techniques does Lazarus Group use?","category": "Group to Techniques"},
    # E. Technique to Groups
    {"id": "E1", "query": "Which groups use T1003?",    "category": "Technique to Groups"},
    {"id": "E2", "query": "Who uses T1055?",            "category": "Technique to Groups"},
    {"id": "E3", "query": "Which groups use T1110.003?","category": "Technique to Groups"},
    {"id": "E4", "query": "Which groups use T1195.002?","category": "Technique to Groups"},
    {"id": "E5", "query": "Which groups use T1021.001?","category": "Technique to Groups"},
    # F. Mitigation Lookup
    {"id": "F1", "query": "How can T1003 be mitigated?",            "category": "Mitigation Lookup"},
    {"id": "F2", "query": "How can T1055 be mitigated?",            "category": "Mitigation Lookup"},
    {"id": "F3", "query": "What mitigations exist for T1110.003?",  "category": "Mitigation Lookup"},
    {"id": "F4", "query": "How can Password Spraying be mitigated?","category": "Mitigation Lookup"},
    {"id": "F5", "query": "How can Process Injection be mitigated?","category": "Mitigation Lookup"},
    # G. Tactic-Filtered Group Queries
    {"id": "G1", "query": "What credential access techniques does APT29 use?",      "category": "Tactic-Filtered Group Query"},
    {"id": "G2", "query": "What persistence techniques does APT29 use?",            "category": "Tactic-Filtered Group Query"},
    {"id": "G3", "query": "What discovery techniques does Lazarus Group use?",      "category": "Tactic-Filtered Group Query"},
    {"id": "G4", "query": "What lateral movement techniques does FIN7 use?",        "category": "Tactic-Filtered Group Query"},
    {"id": "G5", "query": "What execution techniques does APT33 use?",              "category": "Tactic-Filtered Group Query"},
    # H. Analyst Semantic Queries
    {"id": "H1", "query": "How do attackers dump credentials?",                             "category": "Analyst Semantic Query"},
    {"id": "H2", "query": "Explain credential dumping to a SOC analyst.",                   "category": "Analyst Semantic Query"},
    {"id": "H3", "query": "Why is password spraying dangerous?",                            "category": "Analyst Semantic Query"},
    {"id": "H4", "query": "What should defenders monitor for Process Injection?",           "category": "Analyst Semantic Query"},
    {"id": "H5", "query": "What indicators suggest PowerShell abuse?",                      "category": "Analyst Semantic Query"},
    {"id": "H6", "query": "What attack chain would APT29 likely use after initial access?", "category": "Analyst Semantic Query"}
]
print(f"Matrix initialized with {len(QUERIES)} benchmark items.")

In [ ]:
def generate_base_response(prompt, max_new_tokens=512):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None
        )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

base_results = []
print("Executing Parametric Control Run (No Context)...")

for item in QUERIES:
    print(f"Processing Control [{item['id']}]")
    response = generate_base_response(item["query"])
    base_results.append({
        "id": item["id"],
        "query": item["query"],
        "category": item["category"],
        "response": response
    })

df_base = pd.DataFrame(base_results)
df_base.to_csv("qwen_raft_1743_results.csv", index=False)
print("Phase 1 complete. Baseline array logs saved to 'qwen_raft_1743_results.csv'.")

In [ ]:
def generate_raft_response(query, max_new_tokens=512):
    # Retrieve contextual assets via indexed storage vectors
    context, router = chroma_rag.retrieve(query)

    # Reconstruct strict structural target boundaries
    system_prompt = (
        "You are Qwen-ATLAS, an expert threat intelligence analyst specializing in "
        "MITRE ATT&CK. You will be given retrieved ATT&CK context documents. "
        "Answer using the provided context as your primary source. "
        "Cite technique IDs and group names explicitly. "
        "If the context is insufficient to answer fully, state what is missing."
    )
    user_prompt = f"<context>\n{context}\n</context>\n\nQuestion: {query}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            repetition_penalty=1.15  # Stabilizes multi-hop formatting sequences
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return router, response.strip()

# Initialize deployment testing loop
raft_results = []
print("Executing Production RAFT + RAG Pipeline...")

for item in QUERIES:
    print(f"Routing Pipeline Triggered [{item['id']}]: {item['query']}")
    router, response = generate_raft_response(item["query"])
    
    raft_results.append({
        "id": item["id"],
        "category": item["category"],
        "query": item["query"],
        "router": router,
        "response": response
    })

df_raft = pd.DataFrame(raft_results)
df_raft.to_csv("qwen_raft_rag_results_final.csv", index=False)
print("Pipeline benchmark execution complete. Results saved to 'qwen_raft_rag_results_final.csv'.")